In [ ]:
import pandas as pd
import numpy as np
import os

# ==========================================
# 0. GLOBAL SETUP
# ==========================================
PROJECT_ROOT = r'E:\groteEdeepprofilerdingen\DataDeepprofiler\finaal'
INPUT_CSV = os.path.join(PROJECT_ROOT, "7marchecht", "aggregated_wells_min5.csv") 
OUTPUT_DIR = os.path.join(PROJECT_ROOT, "7marchecht")
CONTROL_LABEL = "no_sgRNA" 

print("Loading raw data...")
df_raw = pd.read_csv(INPUT_CSV)
df_raw.columns = [str(c) for c in df_raw.columns]

# Separate Metadata and Features
metadata_cols = ['Plate', 'Well_ID', 'Treatment', 'Cell_Count']
feature_cols = [c for c in df_raw.columns if c not in metadata_cols]

# ==========================================
# TOOLBOX: FUNCTIONS
# ==========================================

def filter_within_plate_consistency(df, features, top_n_to_keep=500):
    ctrls = df[df['Treatment'] == CONTROL_LABEL]
    within_plate_variation = ctrls.groupby('Plate')[features].std().mean()
    consistent_features = within_plate_variation.sort_values(ascending=True).head(top_n_to_keep).index.tolist()
    return consistent_features

def filter_across_plate_stability(df, features, top_n_to_keep=300):
    ctrls = df[df['Treatment'] == CONTROL_LABEL]
    plate_medians = ctrls.groupby('Plate')[features].median()

    tp_batch_noises = []
    for tp in ['T0', 'T1', 'T2']:
        tp_plates = [p for p in plate_medians.index if p.endswith(tp)]
        if len(tp_plates) > 1:
            noise = plate_medians.loc[tp_plates].std()
            tp_batch_noises.append(noise)
    
    if not tp_batch_noises:
        return features 
        
    total_batch_noise = pd.concat(tp_batch_noises, axis=1).mean(axis=1)
    
    # Selection based on batch noise ranking
    stable_features = total_batch_noise.sort_values(ascending=True).head(top_n_to_keep).index.tolist()
    return stable_features

def filter_redundancy(df, features, correlation_threshold=0.9):
    corr_matrix = df[features].corr().abs()
    upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
    to_drop = [column for column in upper.columns if any(upper[column] > correlation_threshold)]
    final_features = [f for f in features if f not in to_drop]
    return final_features

# ==========================================
# EXECUTION PIPELINE
# ==========================================

# --- STEP 0: GLOBAL VARIANCE FILTER ---
# Removes features that are flat/near-zero across the whole experiment first
print(f"\nRunning Step 0: Global Variance Filter (std > 0.01)...")
initial_std = df_raw[feature_cols].std()
active_features = initial_std[initial_std > 0.01].index.tolist()
print(f"Removed {len(feature_cols) - len(active_features)} low-variance features.")

# --- STEP 1: WITHIN-PLATE CONSISTENCY ---
print(f"Running Step 1: Within-Plate Consistency (Filtering to top 3000)...")
step1_features = filter_within_plate_consistency(df_raw, active_features, top_n_to_keep=3000)
df_step1 = df_raw[metadata_cols + step1_features]

# --- STEP 2: ACROSS-PLATE STABILITY ---
print(f"Running Step 2: Across-Plate Stability (Filtering to top 200)...")
step2_features = filter_across_plate_stability(df_step1, step1_features, top_n_to_keep=200)
df_step2 = df_step1[metadata_cols + step2_features]

# --- STEP 3: REDUNDANCY REMOVAL ---
print(f"Running Step 3: Redundancy Filter (Threshold 0.9)...")
final_feature_list = filter_redundancy(df_step2, step2_features, correlation_threshold=0.9)

# ==========================================
# FINAL SAVE
# ==========================================
df_final = df_step2[metadata_cols + final_feature_list]
output_path = os.path.join(OUTPUT_DIR, "vettedcellcounts5_9march_reordered.csv")
df_final.to_csv(output_path, index=False)

print("\n" + "="*40)
print(f"WORKFLOW COMPLETE")
print(f"Original features: {len(feature_cols)}")
print(f"Active features (Step 0): {len(active_features)}")
print(f"Final feature count: {len(final_feature_list)}")
print(f"Saved to: {output_path}")
print("="*40)

In [ ]:
#thuis:
import pandas as pd
import numpy as np
import umap
import os
from sklearn.preprocessing import StandardScaler
import plotly.express as px

# 1. SETUP
PROJECT_ROOT = r'E:\groteEdeepprofilerdingen\DataDeepprofiler\finaal'
# Using the updated filename from the vetting script
file_path = os.path.join(PROJECT_ROOT,"7marchecht","vettedcellcounts5_9march_reordered.csv")
df = pd.read_csv(file_path)

OUTPUT_DIR = os.path.join(PROJECT_ROOT,"8marchecht", "UMAP_5reoredered9march")

if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)

# Ensure column names are strings
df.columns = [str(c) for c in df.columns]

# 2. SELECTION
SELECTED_PLATES = ["PLATE1_T0","PLATE1_T1","PLATE1_T2","PLATE2_T0","PLATE2_T1","PLATE2_T2",
                   "PLATE3_T0","PLATE3_T1","PLATE3_T2","PLATE4_T0","PLATE4_T1","PLATE4_T2",
                   "PLATE5_T0","PLATE5_T1","PLATE5_T2"]

if SELECTED_PLATES:
    df = df[df['Plate'].isin(SELECTED_PLATES)].copy()

# 3. DEFINE CHANNELS
# FIXED: We identify features by checking if the column name is purely numeric
# This excludes 'Plate', 'Well_ID', 'Treatment', 'Cell_Count', and 'Type'
vetted_features = [c for c in df.columns if c.isdigit()]

def get_channel_features(start, end):
    return [f for f in vetted_features if start <= int(f) < end]

plot_configs = [
    {"name": "All Vetted Channels", "indices": vetted_features},
    {"name": "Channel 1", "indices": get_channel_features(0, 1280)},
    {"name": "Channel 2", "indices": get_channel_features(1280, 2560)},
    {"name": "Channel 3", "indices": get_channel_features(2560, 3840)},
    {"name": "Channel 4", "indices": get_channel_features(3840, 5120)},
    {"name": "Channel 5", "indices": get_channel_features(5120, 6400)}
]

# Create a 'Type' column for distinct shapes
df['Type'] = df['Treatment'].apply(lambda x: 'Control' if x.lower() in ["no_sgrna", "nosgrna"] else 'Mutant')

# 4. RUN UMAP FOR EACH CHANNEL
for config in plot_configs:
    if not config['indices']:
        print(f"Skipping {config['name']} (0 features).")
        continue
    
    print(f"Processing interactive UMAP for: {config['name']} ({len(config['indices'])} features)...")
    
    # Scale and Reduce
    X = df[config['indices']].values
    X_scaled = StandardScaler().fit_transform(X)
    
    # UMAP parameters - n_neighbors affects local vs global structure
    reducer = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=42)
    embedding = reducer.fit_transform(X_scaled)
    
    # Temporarily add coordinates to a plotting dataframe
    df_plot = df.copy()
    df_plot['UMAP1'] = embedding[:, 0]
    df_plot['UMAP2'] = embedding[:, 1]
    
    # Create the figure
    fig = px.scatter(
        df_plot, 
        x='UMAP1', 
        y='UMAP2', 
        color='Plate',
        symbol='Type',
        hover_name='Treatment',
        # ADDED: 'Cell_Count' to hover_data so you can check quality during exploration
        hover_data={
            'Plate': True, 
            'Well_ID': True, 
            'Cell_Count': True, 
            'UMAP1': False, 
            'UMAP2': False
        },
        title=f"UMAP: {config['name']} ({len(config['indices'])} features)",
        template='plotly_white',
        symbol_map={'Control': 'square', 'Mutant': 'circle'}
    )
    
    # Tweak visual density
    fig.update_traces(marker=dict(size=6, opacity=0.7), selector=dict(marker_symbol='circle')) 
    fig.update_traces(marker=dict(size=9, line=dict(width=1.5, color='black')), selector=dict(marker_symbol='square')) 
    
    fig.update_layout(
        width=850, 
        height=850,
        legend_title_text='Plate & Timepoint',
        xaxis=dict(showticklabels=False, title="UMAP 1"),
        yaxis=dict(showticklabels=False, title="UMAP 2")
    )
    
    # Save the interactive HTML
    save_path = os.path.join(OUTPUT_DIR, f"UMAP5_{config['name'].replace(' ', '_')}.html")
    fig.write_html(save_path)
    print(f"Saved: {save_path}")
    
    # Show the plot
    fig.show()

In [ ]:
import pandas as pd
import numpy as np
import umap
import os
from sklearn.preprocessing import StandardScaler
import plotly.express as px
import plotly.graph_objects as go

# --- 1. SETUP ---
PROJECT_ROOT = r'E:\groteEdeepprofilerdingen\DataDeepprofiler\finaal'
OUTPUT_DIR = os.path.join(PROJECT_ROOT, "8marchecht", "UMAP_5_annotated3and4_9marchreoredered")

if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)

file_path = os.path.join(PROJECT_ROOT, "7marchecht", "vettedcellcounts5_9march_reordered.csv")
anno_path = os.path.join(PROJECT_ROOT, "Pathway_annotation.xlsx")

df = pd.read_csv(file_path)
anno_df = pd.read_excel(anno_path)
df.columns = [str(c) for c in df.columns]

# --- 2. MERGE & FALLBACK ANNOTATIONS ---
df = df.merge(anno_df[['Treatment', 'SubtiWiki Annotation 3', 'SubtiWiki Annotation 4']], on='Treatment', how='left')

# FALLBACK: Annotation 4 -> Annotation 3 -> Unknown
df['Effective_Annotation'] = df['SubtiWiki Annotation 4'].fillna(df['SubtiWiki Annotation 3'])
df['Effective_Annotation'] = df['Effective_Annotation'].fillna("Unknown/Other")

df['Timepoint'] = df['Plate'].str.extract(r'_(T\d)')

# Define the Display Category
is_control = df['Treatment'].str.lower().isin(["no_sgrna", "nosgrna"])
df['Display_Category'] = df['Effective_Annotation']

# Set Baseline/Control logic
df.loc[is_control, 'Display_Category'] = 'no_sgrna'
df.loc[(df['Timepoint'] == 'T0') & (~is_control), 'Display_Category'] = 'Baseline (T0)'

# --- 3. DYNAMIC COLOR MAPPING ---
# Combine multiple palettes to ensure maximum distinction between pathways (avoids too many purples)
all_cats = sorted([c for c in df['Display_Category'].unique() if c not in ['no_sgrna', 'Baseline (T0)', 'Unknown/Other']])
standard_colors = px.colors.qualitative.Alphabet + px.colors.qualitative.Dark24 + px.colors.qualitative.Light24

color_map = {cat: standard_colors[i % len(standard_colors)] for i, cat in enumerate(all_cats)}
color_map['no_sgrna'] = 'lightgrey'
color_map['Baseline (T0)'] = '#D3D3D3' # Static light grey
color_map['Unknown/Other'] = 'black'

# --- 4. EXECUTION FOR ALL CHANNELS ---
vetted_features = [c for c in df.columns if c.isdigit()]

def get_channel_features(start, end):
    return [f for f in vetted_features if start <= int(f) < end]

plot_configs = [
    {"name": "All_Vetted_Channels", "indices": vetted_features},
    {"name": "Channel_1", "indices": get_channel_features(0, 1280)},
    {"name": "Channel_2", "indices": get_channel_features(1280, 2560)},
    {"name": "Channel_3", "indices": get_channel_features(2560, 3840)},
    {"name": "Channel_4", "indices": get_channel_features(3840, 5120)},
    {"name": "Channel_5", "indices": get_channel_features(5120, 6400)}
]

for config in plot_configs:
    if not config['indices']: continue
    
    print(f"Processing: {config['name']} ({len(config['indices'])} features)...")
    X_scaled = StandardScaler().fit_transform(df[config['indices']].values)
    embedding = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=42).fit_transform(X_scaled)
    df['UMAP1'], df['UMAP2'] = embedding[:, 0], embedding[:, 1]
    
    # Define shapes
    df['Point_Shape'] = df['Timepoint'].map({"T0": "circle", "T1": "x", "T2": "circle"})
    df.loc[is_control, 'Point_Shape'] = 'square'

    fig = px.scatter(
        df, x='UMAP1', y='UMAP2', 
        color='Display_Category',
        symbol='Point_Shape',
        symbol_map={"circle": "circle", "x": "x", "square": "square"},
        hover_name='Treatment',
        hover_data=['Plate', 'Effective_Annotation'],
        title=f"Pathway Overlay: {config['name'].replace('_', ' ')}",
        color_discrete_map=color_map,
        template='plotly_white'
    )
    
    # Legend De-duplication
    seen_pathways = set()
    fig.for_each_trace(lambda t: (
        t.update(showlegend=False) if t.name.split(",")[0] in seen_pathways 
        else (seen_pathways.add(t.name.split(",")[0]), t.update(name=t.name.split(",")[0]))
    ))

    # Styling
    fig.update_traces(marker=dict(opacity=1.0)) 
    fig.update_traces(marker=dict(size=5), selector=dict(marker_symbol='circle'))
    fig.update_traces(marker=dict(size=7), selector=dict(marker_symbol='x'))
    fig.update_traces(marker=dict(size=9, line=dict(width=1, color='black')), selector=dict(marker_symbol='square'))
    
    # --- AXIS AND LAYOUT FIX ---
    fig.update_layout(
        width=1300, height=850,
        legend_title_text='Pathway / Group',
        annotations=[
            dict(
                text="<b>Key:</b> Square = no_sgrna | Cross (x) = T1 | Dot (●) = T2/T0",
                showarrow=False, xref="paper", yref="paper",
                x=0.5, y=1.07, font=dict(size=14),
                bgcolor="white", bordercolor="black", borderwidth=1
            )
        ],
        xaxis=dict(
            title="UMAP 1", 
            showline=True, linewidth=2, linecolor='black', mirror=False, 
            showgrid=False, zeroline=False # REMOVED LIGHT GREY AXIS
        ),
        yaxis=dict(
            title="UMAP 2", 
            showline=True, linewidth=2, linecolor='black', mirror=False, 
            showgrid=False, zeroline=False # REMOVED LIGHT GREY AXIS
        )
    )
    
    # Save
    file_base = f"UMAP_Annotated_{config['name']}"
    fig.write_html(os.path.join(OUTPUT_DIR, f"{file_base}.html"))
    fig.write_image(os.path.join(OUTPUT_DIR, f"{file_base}.svg"))

print(f"Done. All channels processed with clean axes and high-contrast colors.")

In [ ]:
import pandas as pd
import numpy as np
import umap
import os
from sklearn.preprocessing import StandardScaler
import plotly.express as px
import plotly.graph_objects as go

# --- 1. SETUP ---
PROJECT_ROOT = r'E:\groteEdeepprofilerdingen\DataDeepprofiler\finaal'
OUTPUT_DIR = os.path.join(PROJECT_ROOT, "8marchecht", "UMAP_5_Max_Contrast_v3")

if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)

file_path = os.path.join(PROJECT_ROOT, "7marchecht", "vettedcellcounts5_9march_reordered.csv")
anno_path = os.path.join(PROJECT_ROOT, "Pathway_annotation.xlsx")

print("Loading data...")
df_raw = pd.read_csv(file_path)
anno_df = pd.read_excel(anno_path)
df_raw.columns = [str(c) for c in df_raw.columns]

# --- 2. GLOBAL VARIANCE FILTER ---
metadata_cols = ['Plate', 'Well_ID', 'Treatment', 'Cell_Count']
feature_cols = [c for c in df_raw.columns if c.isdigit()]
initial_std = df_raw[feature_cols].std()
active_features = initial_std[initial_std > 0.01].index.tolist()
df = df_raw[metadata_cols + active_features].copy()

# --- 3. MERGE & CATEGORIZATION ---
df = df.merge(anno_df[['Treatment', 'SubtiWiki Annotation 3', 'SubtiWiki Annotation 4']], on='Treatment', how='left')
df['Effective_Annotation'] = df['SubtiWiki Annotation 4'].fillna(df['SubtiWiki Annotation 3']).fillna("Unknown/Other")
df['Timepoint'] = df['Plate'].str.extract(r'_(T\d)')

is_control = df['Treatment'].str.lower().isin(["no_sgrna", "nosgrna"])
df['Display_Category'] = df['Effective_Annotation']
df.loc[is_control, 'Display_Category'] = 'no_sgrna'
df.loc[(df['Timepoint'] == 'T0') & (~is_control), 'Display_Category'] = 'Baseline (T0)'

# --- 4. GOLDEN ANGLE COLOR MAPPING ---
# This ensures maximum perceptual distance between consecutive categories
all_cats = sorted([c for c in df['Display_Category'].unique() if c not in ['no_sgrna', 'Baseline (T0)', 'Unknown/Other']])
num_cats = len(all_cats)

if num_cats > 0:
    # We sample 256 colors from a high-quality spectrum
    base_palette = px.colors.sample_colorscale("Turbo", [i/255 for i in range(256)])
    
    # Use the Golden Angle (~137.5 degrees converted to index steps) 
    # to jump through the palette so neighbors are always far apart
    golden_ratio_conjugate = 0.618033988749895
    h_values = [(i * golden_ratio_conjugate) % 1 for i in range(num_cats)]
    
    # Map these "jumps" to the palette
    max_contrast_palette = [base_palette[int(h * 255)] for h in h_values]
    
    color_map = {cat: max_contrast_palette[i] for i, cat in enumerate(all_cats)}
else:
    color_map = {}

color_map['no_sgrna'] = '#EBEBEB'
color_map['Baseline (T0)'] = '#B0B0B0'
color_map['Unknown/Other'] = '#222222'

# --- 5. EXECUTION ---
vetted_features = [c for c in df.columns if c.isdigit()]

def get_channel_features(start, end):
    return [f for f in vetted_features if start <= int(f) < end]

plot_configs = [
    {"name": "All_Vetted_Channels", "indices": vetted_features},
    {"name": "Channel_1", "indices": get_channel_features(0, 1280)},
    {"name": "Channel_2", "indices": get_channel_features(1280, 2560)},
    {"name": "Channel_3", "indices": get_channel_features(2560, 3840)},
    {"name": "Channel_4", "indices": get_channel_features(3840, 5120)},
    {"name": "Channel_5", "indices": get_channel_features(5120, 6400)}
]

for config in plot_configs:
    if not config['indices']: continue
    
    print(f"Processing: {config['name']}...")
    X_scaled = StandardScaler().fit_transform(df[config['indices']].values)
    embedding = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=42).fit_transform(X_scaled)
    df['UMAP1'], df['UMAP2'] = embedding[:, 0], embedding[:, 1]
    
    df['Point_Shape'] = df['Timepoint'].map({"T0": "circle", "T1": "x", "T2": "circle"})
    df.loc[is_control, 'Point_Shape'] = 'square'

    fig = px.scatter(
        df, x='UMAP1', y='UMAP2', 
        color='Display_Category',
        symbol='Point_Shape',
        symbol_map={"circle": "circle", "x": "x", "square": "square"},
        hover_name='Treatment',
        hover_data=['Plate', 'Effective_Annotation'],
        title=f"Pathway Overlay: {config['name'].replace('_', ' ')}",
        color_discrete_map=color_map,
        template='plotly_white'
    )
    
    seen_pathways = set()
    fig.for_each_trace(lambda t: (
        t.update(showlegend=False) if t.name.split(",")[0] in seen_pathways 
        else (seen_pathways.add(t.name.split(",")[0]), t.update(name=t.name.split(",")[0]))
    ))

    # Slightly higher opacity and black outlines for points to make colors pop
    fig.update_traces(marker=dict(opacity=0.9, line=dict(width=0.5, color='white'))) 
    fig.update_traces(marker=dict(size=5), selector=dict(marker_symbol='circle'))
    fig.update_traces(marker=dict(size=7), selector=dict(marker_symbol='x'))
    fig.update_traces(marker=dict(size=10, line=dict(width=1.5, color='black')), selector=dict(marker_symbol='square'))
    
    fig.update_layout(
        width=1400, height=900,
        legend_title_text='Pathway / Group',
        xaxis=dict(title="UMAP 1", showline=True, linewidth=2, linecolor='black', showgrid=False),
        yaxis=dict(title="UMAP 2", showline=True, linewidth=2, linecolor='black', showgrid=False)
    )
    
    file_base = f"UMAP_Annotated_{config['name']}"
    fig.write_html(os.path.join(OUTPUT_DIR, f"{file_base}.html"))

print(f"Done. Golden-angle sampling applied for maximum distinction.")

In [ ]:
import pandas as pd
import numpy as np
import umap
import os
from sklearn.preprocessing import StandardScaler
import plotly.express as px
import plotly.graph_objects as go

# --- 1. SETUP ---
PROJECT_ROOT = r'E:\groteEdeepprofilerdingen\DataDeepprofiler\finaal'
OUTPUT_DIR = os.path.join(PROJECT_ROOT, "8marchecht", "UMAP_5_annotated3and4names_GoldenAngle")
COORD_DIR = os.path.join(OUTPUT_DIR, "Coordinates")

for folder in [OUTPUT_DIR, COORD_DIR]:
    if not os.path.exists(folder):
        os.makedirs(folder)

file_path = os.path.join(PROJECT_ROOT, "7marchecht", "vettedcellcounts5_9march_reordered.csv")
anno_path = os.path.join(PROJECT_ROOT, "Pathway_annotation.xlsx")

print("Loading data...")
df = pd.read_csv(file_path)
anno_df = pd.read_excel(anno_path)
df.columns = [str(c) for c in df.columns]

# --- 2. PLATE/TIME FILTERING ---
SELECTED_PLATES = [] 

if SELECTED_PLATES:
    df = df[df['Plate'].isin(SELECTED_PLATES)].copy()

# --- 3. MERGE & FALLBACK ANNOTATIONS ---
df = df.merge(anno_df[['Treatment', 'SubtiWiki Annotation 3', 'SubtiWiki Annotation 4']], on='Treatment', how='left')
df['Effective_Annotation'] = df['SubtiWiki Annotation 4'].fillna(df['SubtiWiki Annotation 3']).fillna("Unknown/Other")
df['Timepoint'] = df['Plate'].str.extract(r'_(T\d)')

# Define the Display Category
is_control = df['Treatment'].str.lower().isin(["no_sgrna", "nosgrna"])
df['Display_Category'] = df['Effective_Annotation']
df.loc[is_control, 'Display_Category'] = 'no_sgrna'
df.loc[(df['Timepoint'] == 'T0') & (~is_control), 'Display_Category'] = 'Baseline (T0)'

# --- 4. DYNAMIC COLOR MAPPING (GOLDEN ANGLE TURBO) ---
all_cats = sorted([c for c in df['Display_Category'].unique() if c not in ['no_sgrna', 'Baseline (T0)', 'Unknown/Other']])
num_cats = len(all_cats)

if num_cats > 0:
    # Sample colors from the Turbo spectrum
    base_palette = px.colors.sample_colorscale("Turbo", [i/255 for i in range(256)])
    
    # Golden angle jump to maximize contrast between consecutive sorted names
    golden_ratio_conjugate = 0.618033988749895
    h_values = [(i * golden_ratio_conjugate) % 1 for i in range(num_cats)]
    
    max_contrast_palette = [base_palette[int(h * 255)] for h in h_values]
    color_map = {cat: max_contrast_palette[i] for i, cat in enumerate(all_cats)}
else:
    color_map = {}

# Fixed colors for controls and baseline
color_map['no_sgrna'] = '#EBEBEB'     # Very light grey
color_map['Baseline (T0)'] = '#B0B0B0' # Medium grey
color_map['Unknown/Other'] = '#222222' # Charcoal/Black

# --- 5. EXECUTION ---
vetted_features = [c for c in df.columns if c.isdigit()]
def get_channel_features(start, end):
    return [f for f in vetted_features if start <= int(f) < end]

plot_configs = [
    {"name": "All_Vetted_Channels", "indices": vetted_features},
    {"name": "Channel_1", "indices": get_channel_features(0, 1280)},
    {"name": "Channel_2", "indices": get_channel_features(1280, 2560)},
    {"name": "Channel_3", "indices": get_channel_features(2560, 3840)},
    {"name": "Channel_4", "indices": get_channel_features(3840, 5120)},
    {"name": "Channel_5", "indices": get_channel_features(5120, 6400)}
]

for config in plot_configs:
    if not config['indices']: continue
    
    print(f"Processing: {config['name']}...")
    X_scaled = StandardScaler().fit_transform(df[config['indices']].values)
    
    embedding = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=42).fit_transform(X_scaled)
    df['UMAP1'], df['UMAP2'] = embedding[:, 0], embedding[:, 1]
    
    # Save coordinates
    coord_filename = f"Coordinates_{config['name']}.csv"
    df[['Plate', 'Well_ID', 'Treatment', 'Display_Category', 'UMAP1', 'UMAP2']].to_csv(os.path.join(COORD_DIR, coord_filename), index=False)
    
    # --- 6. PLOTTING ---
    df['Point_Shape'] = df['Timepoint'].map({"T0": "circle", "T1": "x", "T2": "circle"})
    df.loc[is_control, 'Point_Shape'] = 'square'

    fig = px.scatter(
        df, x='UMAP1', y='UMAP2', 
        color='Display_Category',
        symbol='Point_Shape',
        text='Treatment', 
        symbol_map={"circle": "circle", "x": "x", "square": "square"},
        hover_name='Treatment',
        hover_data={'Plate': True, 'Well_ID': True, 'Effective_Annotation': True, 'UMAP1': False, 'UMAP2': False},
        title=f"Pathway Overlay: {config['name'].replace('_', ' ')}",
        color_discrete_map=color_map,
        template='plotly_white'
    )
    
    seen_pathways = set()
    fig.for_each_trace(lambda t: (
        t.update(showlegend=False) if t.name.split(",")[0] in seen_pathways 
        else (seen_pathways.add(t.name.split(",")[0]), t.update(name=t.name.split(",")[0]))
    ))

    # Labels size 16
    fig.update_traces(
        mode='markers+text', 
        textposition='top center', 
        textfont=dict(size=16, color='black'), 
        marker=dict(opacity=1.0, line=dict(width=0.5, color='white')) # Added thin outline to help distinguish clusters
    )
    
    # Marker sizes (Twice the original size as requested)
    fig.update_traces(marker=dict(size=10), selector=dict(marker_symbol='circle')) 
    fig.update_traces(marker=dict(size=14), selector=dict(marker_symbol='x'))      
    fig.update_traces(marker=dict(size=18, line=dict(width=1.5, color='black')), selector=dict(marker_symbol='square')) 
    
    # Axis titles size 18
    fig.update_layout(
        width=1600, height=1000,
        legend_title_text='Pathway / Group',
        annotations=[
            dict(
                text="<b>Key:</b> Square = no_sgrna | Cross (x) = T1 | Dot (●) = T2/T0",
                showarrow=False, xref="paper", yref="paper",
                x=0.5, y=1.07, font=dict(size=14),
                bgcolor="white", bordercolor="black", borderwidth=1
            )
        ],
        xaxis=dict(
            title=dict(text="UMAP 1", font=dict(size=18)), 
            showline=True, linewidth=2, linecolor='black', 
            mirror=False, showgrid=False, zeroline=False
        ),
        yaxis=dict(
            title=dict(text="UMAP 2", font=dict(size=18)), 
            showline=True, linewidth=2, linecolor='black', 
            mirror=False, showgrid=False, zeroline=False
        )
    )
    
    file_base = f"UMAP_Annotated_{config['name']}"
    fig.write_html(os.path.join(OUTPUT_DIR, f"{file_base}.html"))
    fig.write_image(os.path.join(OUTPUT_DIR, f"{file_base}.svg"))

print(f"Done. Colors are mathematically distinct and markers are scaled up.")

In [ ]:
import pandas as pd
import numpy as np
import umap
import os
from sklearn.preprocessing import StandardScaler
import plotly.express as px
import plotly.graph_objects as go

# --- 1. SETUP ---
PROJECT_ROOT = r'E:\groteEdeepprofilerdingen\DataDeepprofiler\finaal'
OUTPUT_DIR = os.path.join(PROJECT_ROOT, "8marchecht", "UMAP_5_annotated3and4names_T2")
COORD_DIR = os.path.join(OUTPUT_DIR, "Coordinates")

for folder in [OUTPUT_DIR, COORD_DIR]:
    if not os.path.exists(folder):
        os.makedirs(folder)

file_path = os.path.join(PROJECT_ROOT, "7marchecht", "vettedcellcounts5_9march_reordered.csv")
anno_path = os.path.join(PROJECT_ROOT, "Pathway_annotation.xlsx")

print("Loading data...")
df = pd.read_csv(file_path)
anno_df = pd.read_excel(anno_path)
df.columns = [str(c) for c in df.columns]

# --- 2. PLATE/TIME FILTERING ---
SELECTED_PLATES = ["PLATE1_T2", "PLATE2_T2", "PLATE3_T2", "PLATE4_T2", "PLATE5_T2"]  # Only T2 timepoints

if SELECTED_PLATES:
    df = df[df['Plate'].isin(SELECTED_PLATES)].copy()

# --- 3. MERGE & FALLBACK ANNOTATIONS ---
df = df.merge(anno_df[['Treatment', 'SubtiWiki Annotation 3', 'SubtiWiki Annotation 4']], on='Treatment', how='left')
df['Effective_Annotation'] = df['SubtiWiki Annotation 4'].fillna(df['SubtiWiki Annotation 3']).fillna("Unknown/Other")
df['Timepoint'] = df['Plate'].str.extract(r'_(T\d)')

# Define the Display Category
is_control = df['Treatment'].str.lower().isin(["no_sgrna", "nosgrna"])
df['Display_Category'] = df['Effective_Annotation']
df.loc[is_control, 'Display_Category'] = 'no_sgrna'
df.loc[(df['Timepoint'] == 'T0') & (~is_control), 'Display_Category'] = 'Baseline (T0)'

# --- 4. DYNAMIC COLOR MAPPING (GOLDEN ANGLE TURBO) ---
all_cats = sorted([c for c in df['Display_Category'].unique() if c not in ['no_sgrna', 'Baseline (T0)', 'Unknown/Other']])
num_cats = len(all_cats)

if num_cats > 0:
    # Sample colors from the Turbo spectrum
    base_palette = px.colors.sample_colorscale("Turbo", [i/255 for i in range(256)])
    
    # Golden angle jump to maximize contrast between consecutive sorted names
    golden_ratio_conjugate = 0.618033988749895
    h_values = [(i * golden_ratio_conjugate) % 1 for i in range(num_cats)]
    
    max_contrast_palette = [base_palette[int(h * 255)] for h in h_values]
    color_map = {cat: max_contrast_palette[i] for i, cat in enumerate(all_cats)}
else:
    color_map = {}

# Fixed colors for controls and baseline
color_map['no_sgrna'] = '#EBEBEB'     # Very light grey
color_map['Baseline (T0)'] = '#B0B0B0' # Medium grey
color_map['Unknown/Other'] = '#222222' # Charcoal/Black

# --- 5. EXECUTION ---
vetted_features = [c for c in df.columns if c.isdigit()]
def get_channel_features(start, end):
    return [f for f in vetted_features if start <= int(f) < end]

plot_configs = [
    {"name": "All_Vetted_Channels", "indices": vetted_features},
    {"name": "Channel_1", "indices": get_channel_features(0, 1280)},
    {"name": "Channel_2", "indices": get_channel_features(1280, 2560)},
    {"name": "Channel_3", "indices": get_channel_features(2560, 3840)},
    {"name": "Channel_4", "indices": get_channel_features(3840, 5120)},
    {"name": "Channel_5", "indices": get_channel_features(5120, 6400)}
]

for config in plot_configs:
    if not config['indices']: continue
    
    print(f"Processing: {config['name']}...")
    X_scaled = StandardScaler().fit_transform(df[config['indices']].values)
    
    embedding = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=42).fit_transform(X_scaled)
    df['UMAP1'], df['UMAP2'] = embedding[:, 0], embedding[:, 1]
    
    # Save coordinates
    coord_filename = f"Coordinates_{config['name']}.csv"
    df[['Plate', 'Well_ID', 'Treatment', 'Display_Category', 'UMAP1', 'UMAP2']].to_csv(os.path.join(COORD_DIR, coord_filename), index=False)
    
    # --- 6. PLOTTING ---
    df['Point_Shape'] = df['Timepoint'].map({"T0": "circle", "T1": "x", "T2": "circle"})
    df.loc[is_control, 'Point_Shape'] = 'square'

    fig = px.scatter(
        df, x='UMAP1', y='UMAP2', 
        color='Display_Category',
        symbol='Point_Shape',
        text='Treatment', 
        symbol_map={"circle": "circle", "x": "x", "square": "square"},
        hover_name='Treatment',
        hover_data={'Plate': True, 'Well_ID': True, 'Effective_Annotation': True, 'UMAP1': False, 'UMAP2': False},
        title=f"Pathway Overlay: {config['name'].replace('_', ' ')}",
        color_discrete_map=color_map,
        template='plotly_white'
    )
    
    seen_pathways = set()
    fig.for_each_trace(lambda t: (
        t.update(showlegend=False) if t.name.split(",")[0] in seen_pathways 
        else (seen_pathways.add(t.name.split(",")[0]), t.update(name=t.name.split(",")[0]))
    ))

    # Labels size 16
    fig.update_traces(
        mode='markers+text', 
        textposition='top center', 
        textfont=dict(size=16, color='black'), 
        marker=dict(opacity=1.0, line=dict(width=0.5, color='white')) # Added thin outline to help distinguish clusters
    )
    
    # Marker sizes (Twice the original size as requested)
    fig.update_traces(marker=dict(size=10), selector=dict(marker_symbol='circle')) 
    fig.update_traces(marker=dict(size=14), selector=dict(marker_symbol='x'))      
    fig.update_traces(marker=dict(size=18, line=dict(width=1.5, color='black')), selector=dict(marker_symbol='square')) 
    
    # Axis titles size 18
    fig.update_layout(
        width=1600, height=1000,
        legend_title_text='Pathway / Group',
        annotations=[
            dict(
                text="<b>Key:</b> Square = no_sgrna | Cross (x) = T1 | Dot (●) = T2/T0",
                showarrow=False, xref="paper", yref="paper",
                x=0.5, y=1.07, font=dict(size=14),
                bgcolor="white", bordercolor="black", borderwidth=1
            )
        ],
        xaxis=dict(
            title=dict(text="UMAP 1", font=dict(size=18)), 
            showline=True, linewidth=2, linecolor='black', 
            mirror=False, showgrid=False, zeroline=False
        ),
        yaxis=dict(
            title=dict(text="UMAP 2", font=dict(size=18)), 
            showline=True, linewidth=2, linecolor='black', 
            mirror=False, showgrid=False, zeroline=False
        )
    )
    
    file_base = f"UMAP_Annotated_{config['name']}"
    fig.write_html(os.path.join(OUTPUT_DIR, f"{file_base}.html"))
    fig.write_image(os.path.join(OUTPUT_DIR, f"{file_base}.svg"))

print(f"Done. Colors are mathematically distinct and markers are scaled up.")

In [ ]:
#annotated but no labels

In [ ]:
import pandas as pd
import numpy as np
import umap
import os
from sklearn.preprocessing import StandardScaler
import plotly.express as px
import plotly.graph_objects as go

# --- 1. SETUP ---
PROJECT_ROOT = r'E:\groteEdeepprofilerdingen\DataDeepprofiler\finaal'
# Changed folder name to reflect this is the "No Labels" version
OUTPUT_DIR = os.path.join(PROJECT_ROOT, "8marchecht", "UMAP_5_NoLabels_T0")
COORD_DIR = os.path.join(OUTPUT_DIR, "Coordinates")

for folder in [OUTPUT_DIR, COORD_DIR]:
    if not os.path.exists(folder):
        os.makedirs(folder)

file_path = os.path.join(PROJECT_ROOT, "7marchecht", "vettedcellcounts5_9march_reordered.csv")
anno_path = os.path.join(PROJECT_ROOT, "Pathway_annotation.xlsx")

print("Loading data...")
df = pd.read_csv(file_path)
anno_df = pd.read_excel(anno_path)
df.columns = [str(c) for c in df.columns]

# --- 2. PLATE/TIME FILTERING ---
SELECTED_PLATES = ["PLATE1_T0", "PLATE2_T0", "PLATE3_T0", "PLATE4_T0", "PLATE5_T0"] 

if SELECTED_PLATES:
    df = df[df['Plate'].isin(SELECTED_PLATES)].copy()

# --- 3. MERGE & FALLBACK ANNOTATIONS ---
df = df.merge(anno_df[['Treatment', 'SubtiWiki Annotation 3', 'SubtiWiki Annotation 4']], on='Treatment', how='left')
df['Effective_Annotation'] = df['SubtiWiki Annotation 4'].fillna(df['SubtiWiki Annotation 3']).fillna("Unknown/Other")
df['Timepoint'] = df['Plate'].str.extract(r'_(T\d)')

# Define the Display Category
is_control = df['Treatment'].str.lower().isin(["no_sgrna", "nosgrna"])
df['Display_Category'] = df['Effective_Annotation']
df.loc[is_control, 'Display_Category'] = 'no_sgrna'
df.loc[(df['Timepoint'] == 'T0') & (~is_control), 'Display_Category'] = 'Baseline (T0)'

# --- 4. DYNAMIC COLOR MAPPING (GOLDEN ANGLE TURBO) ---
all_cats = sorted([c for c in df['Display_Category'].unique() if c not in ['no_sgrna', 'Baseline (T0)', 'Unknown/Other']])
num_cats = len(all_cats)

if num_cats > 0:
    base_palette = px.colors.sample_colorscale("Turbo", [i/255 for i in range(256)])
    golden_ratio_conjugate = 0.618033988749895
    h_values = [(i * golden_ratio_conjugate) % 1 for i in range(num_cats)]
    max_contrast_palette = [base_palette[int(h * 255)] for h in h_values]
    color_map = {cat: max_contrast_palette[i] for i, cat in enumerate(all_cats)}
else:
    color_map = {}

color_map['no_sgrna'] = '#EBEBEB'
color_map['Baseline (T0)'] = '#B0B0B0'
color_map['Unknown/Other'] = '#222222'

# --- 5. EXECUTION ---
vetted_features = [c for c in df.columns if c.isdigit()]
def get_channel_features(start, end):
    return [f for f in vetted_features if start <= int(f) < end]

plot_configs = [
    {"name": "All_Vetted_Channels", "indices": vetted_features},
    {"name": "Channel_1", "indices": get_channel_features(0, 1280)},
    {"name": "Channel_2", "indices": get_channel_features(1280, 2560)},
    {"name": "Channel_3", "indices": get_channel_features(2560, 3840)},
    {"name": "Channel_4", "indices": get_channel_features(3840, 5120)},
    {"name": "Channel_5", "indices": get_channel_features(5120, 6400)}
]

for config in plot_configs:
    if not config['indices']: continue
    
    print(f"Processing: {config['name']}...")
    X_scaled = StandardScaler().fit_transform(df[config['indices']].values)
    
    embedding = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=42).fit_transform(X_scaled)
    df['UMAP1'], df['UMAP2'] = embedding[:, 0], embedding[:, 1]
    
    # Save coordinates
    coord_filename = f"Coordinates_{config['name']}.csv"
    df[['Plate', 'Well_ID', 'Treatment', 'Display_Category', 'UMAP1', 'UMAP2']].to_csv(os.path.join(COORD_DIR, coord_filename), index=False)
    
    # --- 6. PLOTTING (MODIFIED FOR NO TEXT LABELS) ---
    df['Point_Shape'] = df['Timepoint'].map({"T0": "circle", "T1": "x", "T2": "circle"})
    df.loc[is_control, 'Point_Shape'] = 'square'

    fig = px.scatter(
        df, x='UMAP1', y='UMAP2', 
        color='Display_Category',
        symbol='Point_Shape',
        # 'text' parameter removed to declutter the graph
        symbol_map={"circle": "circle", "x": "x", "square": "square"},
        hover_name='Treatment',
        hover_data={'Plate': True, 'Well_ID': True, 'Effective_Annotation': True, 'UMAP1': False, 'UMAP2': False},
        title=f"Pathway Overlay (T2 Only): {config['name'].replace('_', ' ')}",
        color_discrete_map=color_map,
        template='plotly_white'
    )
    
    seen_pathways = set()
    fig.for_each_trace(lambda t: (
        t.update(showlegend=False) if t.name.split(",")[0] in seen_pathways 
        else (seen_pathways.add(t.name.split(",")[0]), t.update(name=t.name.split(",")[0]))
    ))

    # mode changed to 'markers' only
    fig.update_traces(
        mode='markers', 
        marker=dict(opacity=1.0, line=dict(width=0.5, color='white'))
    )
    
    # Marker sizes kept large for visibility
    fig.update_traces(marker=dict(size=10), selector=dict(marker_symbol='circle')) 
    fig.update_traces(marker=dict(size=14), selector=dict(marker_symbol='x'))      
    fig.update_traces(marker=dict(size=18, line=dict(width=1.5, color='black')), selector=dict(marker_symbol='square')) 
    
    fig.update_layout(
        width=1600, height=1000,
        legend_title_text='Pathway / Group',
        annotations=[
            dict(
                text="<b>Key:</b> Square = no_sgrna | Cross (x) = T1 | Dot (●) = T2/T0",
                showarrow=False, xref="paper", yref="paper",
                x=0.5, y=1.07, font=dict(size=14),
                bgcolor="white", bordercolor="black", borderwidth=1
            )
        ],
        xaxis=dict(
            title=dict(text="UMAP 1", font=dict(size=18)), 
            showline=True, linewidth=2, linecolor='black', 
            showgrid=False, zeroline=False
        ),
        yaxis=dict(
            title=dict(text="UMAP 2", font=dict(size=18)), 
            showline=True, linewidth=2, linecolor='black', 
            showgrid=False, zeroline=False
        )
    )
    
    file_base = f"UMAP_T2_NoLabels_{config['name']}"
    fig.write_html(os.path.join(OUTPUT_DIR, f"{file_base}.html"))
    fig.write_image(os.path.join(OUTPUT_DIR, f"{file_base}.svg"))

print(f"Done. Graphs generated without mutant labels for a cleaner view.")

In [ ]:
import pandas as pd
import numpy as np
import umap
import os
from sklearn.preprocessing import StandardScaler
import plotly.graph_objects as go
import plotly.express as px

# --- 1. SETUP ---
PROJECT_ROOT = r'E:\groteEdeepprofilerdingen\DataDeepprofiler\finaal'
file_path = os.path.join(PROJECT_ROOT, "7marchecht", "vettedcellcounts5_9march_reordered.csv")
df = pd.read_csv(file_path)

# Separate directories for HTML and SVG
HTML_DIR = os.path.join(PROJECT_ROOT, "8marchecht", "UMAP_preprosessInteractive9march")
SVG_DIR = os.path.join(PROJECT_ROOT, "8marchecht", "UMAP_preprosessVector_Scalable9march")

for folder in [HTML_DIR, SVG_DIR]:
    if not os.path.exists(folder):
        os.makedirs(folder)

df.columns = [str(c) for c in df.columns]

# Data Parsing
df['Timepoint'] = df['Plate'].str.extract(r'(T\d+)')
df['Type'] = df['Treatment'].apply(lambda x: 'Control' if str(x).lower() in ["no_sgrna", "nosgrna"] else 'Mutant')

# --- 2. COLOR SCHEMES ---
unique_combos = sorted(df['Plate'].unique())
turbo_colors = px.colors.sample_colorscale("Turbo", [i/(len(unique_combos)-1) for i in range(len(unique_combos))])
combo_color_map = {combo: turbo_colors[i] for i, combo in enumerate(unique_combos)}

time_colors = {
    'T0': '#A9D1FF', 
    'T1': '#2A7FFF', 
    'T2': "#1647AA"  
}

# --- 3. CHANNEL SELECTION ---
vetted_features = [c for c in df.columns if c.isdigit()]
def get_channel_features(start, end):
    return [f for f in vetted_features if start <= int(f) < end]

plot_configs = [
    {"name": "All Vetted Channels", "indices": vetted_features},
    {"name": "Channel 1", "indices": get_channel_features(0, 1280)},
    {"name": "Channel 2", "indices": get_channel_features(1280, 2560)},
    {"name": "Channel 3", "indices": get_channel_features(2560, 3840)},
    {"name": "Channel 4", "indices": get_channel_features(3840, 5120)},
    {"name": "Channel 5", "indices": get_channel_features(5120, 6400)}
]

# --- 4. EXECUTION ---
for config in plot_configs:
    if not config['indices']: continue
    print(f"Generating Plots and SVGs for: {config['name']}...")
    
    X_scaled = StandardScaler().fit_transform(df[config['indices']].values)
    embedding = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=42).fit_transform(X_scaled)
    
    df_plot = df.copy()
    df_plot['UMAP1'], df_plot['UMAP2'] = embedding[:, 0], embedding[:, 1]

    # --- PLOT LOOP (Plate & Time) ---
    modes = [
        ('PLATE', unique_combos, combo_color_map, 'Plate-Time View'),
        ('TIME', sorted(df_plot['Timepoint'].unique()), time_colors, 'Timepoint View')
    ]

    for mode_name, groups, color_map, title_prefix in modes:
        fig = go.Figure()
        
        for group in groups:
            for t_type in ['Mutant', 'Control']:
                # Determine filtering logic based on mode
                mask = (df_plot['Plate' if mode_name == 'PLATE' else 'Timepoint'] == group) & (df_plot['Type'] == t_type)
                curr = df_plot[mask]
                if curr.empty: continue
                
                color = color_map[group]
                
                fig.add_trace(go.Scatter(
                    x=curr['UMAP1'], y=curr['UMAP2'], mode='markers',
                    name=str(group),
                    marker=dict(
                        color=color, 
                        size=8 if t_type == 'Mutant' else 11,
                        symbol='circle' if t_type == 'Mutant' else 'square',
                        line=dict(width=1.0, color='black') if t_type == 'Control' else dict(width=0),
                        opacity=1.0
                    ),
                    customdata=np.stack((curr['Plate'], curr['Well_ID'], curr['Treatment'], curr['Cell_Count']), axis=-1),
                    hovertemplate="<b>%{customdata[2]}</b><br>Plate: %{customdata[0]}<br>Well: %{customdata[1]}<br>Count: %{customdata[3]}<extra></extra>",
                    showlegend=True if t_type == 'Mutant' else False,
                    legendgroup=str(group)
                ))

        fig.update_layout(
            title=f"{title_prefix}: {config['name']}",
            template='plotly_white',
            width=850, height=850,
            xaxis=dict(title="UMAP 1", showgrid=False, showticklabels=False),
            yaxis=dict(title="UMAP 2", showgrid=False, showticklabels=False)
        )

        # 1. Save HTML
        html_name = f"{config['name'].replace(' ', '_')}_{mode_name}.html"
        fig.write_html(os.path.join(HTML_DIR, html_name))
        
        # 2. Save SVG (requires kaleido)
        svg_name = f"{config['name'].replace(' ', '_')}_{mode_name}.svg"
        fig.write_image(os.path.join(SVG_DIR, svg_name))

    print(f"Successfully saved {config['name']} exports.")

print("\nAll files (HTML and SVG) are ready in the output folders.")

In [ ]:
# --- COUNT SUMMARY ---
# Create a temporary 'Status' column for easy counting
df['Status'] = np.where(df['Treatment'].str.lower().isin(["no_sgrna", "nosgrna"]), 'no_sgRNA', 'Mutant')

# Group by Timepoint and Status to get counts
summary_counts = df.groupby(['Timepoint', 'Status']).size().unstack(fill_value=0)

# Add a Total column
summary_counts['Total'] = summary_counts.sum(axis=1)

print("\n" + "="*30)
print("EXPERIMENT SAMPLE SUMMARY")
print("="*30)
print(summary_counts)
print("="*30)

# Optional: Print total unique mutants (excluding controls)
unique_mutants = df[df['Status'] == 'Mutant']['Treatment'].nunique()
print(f"Total Unique Mutant Strains: {unique_mutants}")

In [ ]:
####hieronder uitprobeersels





##########

In [ ]:
import pandas as pd
import numpy as np
import os

# ==========================================
# 0. GLOBAL SETUP
# ==========================================
PROJECT_ROOT = r'E:\groteEdeepprofilerdingen\DataDeepprofiler\finaal'
INPUT_CSV = os.path.join(PROJECT_ROOT, "7marchecht", "aggregated_wells_min5.csv") 
OUTPUT_DIR = os.path.join(PROJECT_ROOT, "7marchecht")
CONTROL_LABEL = "no_sgRNA" 

print("Loading raw data...")
df_raw = pd.read_csv(INPUT_CSV)
df_raw.columns = [str(c) for c in df_raw.columns]

# Separate Metadata and Features
metadata_cols = ['Plate', 'Well_ID', 'Treatment', 'Cell_Count']
feature_cols = [c for c in df_raw.columns if c not in metadata_cols]

# ==========================================
# TOOLBOX: FUNCTIONS
# ==========================================

def filter_within_plate_consistency(df, features, top_n_to_keep=500):
    """Original Step 2: Now performed 1st"""
    ctrls = df[df['Treatment'] == CONTROL_LABEL]
    within_plate_variation = ctrls.groupby('Plate')[features].std().mean()
    consistent_features = within_plate_variation.sort_values(ascending=True).head(top_n_to_keep).index.tolist()
    return consistent_features

def filter_across_plate_stability(df, features, top_n_to_keep=300):
    """Original Step 1: Now performed 2nd"""
    ctrls = df[df['Treatment'] == CONTROL_LABEL]
    plate_medians = ctrls.groupby('Plate')[features].median()

    tp_batch_noises = []
    for tp in ['T0', 'T1', 'T2']:
        tp_plates = [p for p in plate_medians.index if p.endswith(tp)]
        if len(tp_plates) > 1:
            noise = plate_medians.loc[tp_plates].std()
            tp_batch_noises.append(noise)
    
    if not tp_batch_noises:
        return features # Fallback if no timepoints match
        
    total_batch_noise = pd.concat(tp_batch_noises, axis=1).mean(axis=1)
    
    # Variance check (ensure features aren't just flat zeros)
    variance_filter = df[features].std() > 0.01
    
    stable_features = total_batch_noise[variance_filter].sort_values(ascending=True).head(top_n_to_keep).index.tolist()
    return stable_features

def filter_redundancy(df, features, correlation_threshold=0.9):
    """Original Step 4: Now performed 3rd"""
    corr_matrix = df[features].corr().abs()
    upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
    to_drop = [column for column in upper.columns if any(upper[column] > correlation_threshold)]
    final_features = [f for f in features if f not in to_drop]
    return final_features

# ==========================================
# EXECUTION PIPELINE
# ==========================================

# --- NEW STEP 1: WITHIN-PLATE CONSISTENCY ---
print(f"\nRunning Step 1: Within-Plate Consistency (Filtering to top 500)...")
step1_features = filter_within_plate_consistency(df_raw, feature_cols, top_n_to_keep=1000)
df_step1 = df_raw[metadata_cols + step1_features]

# --- NEW STEP 2: ACROSS-PLATE STABILITY ---
print(f"Running Step 2: Across-Plate Stability (Filtering to top 300)...")
step2_features = filter_across_plate_stability(df_step1, step1_features, top_n_to_keep=200)
df_step2 = df_step1[metadata_cols + step2_features]

# --- NEW STEP 3: REDUNDANCY REMOVAL ---
# (Note: Step 3 Convergence is skipped as requested)
print(f"Running Step 3: Redundancy Filter (Threshold 0.9)...")
final_feature_list = filter_redundancy(df_step2, step2_features, correlation_threshold=0.9)

# ==========================================
# FINAL SAVE
# ==========================================
df_final = df_step2[metadata_cols + final_feature_list]
output_path = os.path.join(OUTPUT_DIR, "vettedcellcounts5_9march_reordered.csv")
df_final.to_csv(output_path, index=False)

print("\n" + "="*40)
print(f"WORKFLOW COMPLETE (Reordered: Within -> Across -> Redundancy)")
print(f"Original features: {len(feature_cols)}")
print(f"Final feature count: {len(final_feature_list)}")
print(f"Saved to: {output_path}")
print("="*40)

In [ ]:
#thuis:
import pandas as pd
import numpy as np
import umap
import os
from sklearn.preprocessing import StandardScaler
import plotly.express as px

# 1. SETUP
PROJECT_ROOT = r'E:\groteEdeepprofilerdingen\DataDeepprofiler\finaal'
# Using the updated filename from the vetting script
file_path = os.path.join(PROJECT_ROOT,"7marchecht","vettedcellcounts5.csv")
df = pd.read_csv(file_path)

OUTPUT_DIR = os.path.join(PROJECT_ROOT,"8marchecht", "UMAP_59marchtest")

if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)

# Ensure column names are strings
df.columns = [str(c) for c in df.columns]

# 2. SELECTION
SELECTED_PLATES = ["PLATE1_T0","PLATE1_T1","PLATE1_T2","PLATE2_T0","PLATE2_T1","PLATE2_T2",
                   "PLATE3_T0","PLATE3_T1","PLATE3_T2","PLATE4_T0","PLATE4_T1","PLATE4_T2",
                   "PLATE5_T0","PLATE5_T1","PLATE5_T2"]

if SELECTED_PLATES:
    df = df[df['Plate'].isin(SELECTED_PLATES)].copy()

# 3. DEFINE CHANNELS
# FIXED: We identify features by checking if the column name is purely numeric
# This excludes 'Plate', 'Well_ID', 'Treatment', 'Cell_Count', and 'Type'
vetted_features = [c for c in df.columns if c.isdigit()]

def get_channel_features(start, end):
    return [f for f in vetted_features if start <= int(f) < end]

plot_configs = [
    {"name": "All Vetted Channels", "indices": vetted_features},
    {"name": "Channel 1", "indices": get_channel_features(0, 1280)},
    {"name": "Channel 2", "indices": get_channel_features(1280, 2560)},
    {"name": "Channel 3", "indices": get_channel_features(2560, 3840)},
    {"name": "Channel 4", "indices": get_channel_features(3840, 5120)},
    {"name": "Channel 5", "indices": get_channel_features(5120, 6400)}
]

# Create a 'Type' column for distinct shapes
df['Type'] = df['Treatment'].apply(lambda x: 'Control' if x.lower() in ["no_sgrna", "nosgrna"] else 'Mutant')

# 4. RUN UMAP FOR EACH CHANNEL
for config in plot_configs:
    if not config['indices']:
        print(f"Skipping {config['name']} (0 features).")
        continue
    
    print(f"Processing interactive UMAP for: {config['name']} ({len(config['indices'])} features)...")
    
    # Scale and Reduce
    X = df[config['indices']].values
    X_scaled = StandardScaler().fit_transform(X)
    
    # UMAP parameters - n_neighbors affects local vs global structure
    reducer = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=42)
    embedding = reducer.fit_transform(X_scaled)
    
    # Temporarily add coordinates to a plotting dataframe
    df_plot = df.copy()
    df_plot['UMAP1'] = embedding[:, 0]
    df_plot['UMAP2'] = embedding[:, 1]
    
    # Create the figure
    fig = px.scatter(
        df_plot, 
        x='UMAP1', 
        y='UMAP2', 
        color='Plate',
        symbol='Type',
        hover_name='Treatment',
        # ADDED: 'Cell_Count' to hover_data so you can check quality during exploration
        hover_data={
            'Plate': True, 
            'Well_ID': True, 
            'Cell_Count': True, 
            'UMAP1': False, 
            'UMAP2': False
        },
        title=f"UMAP: {config['name']} ({len(config['indices'])} features)",
        template='plotly_white',
        symbol_map={'Control': 'square', 'Mutant': 'circle'}
    )
    
    # Tweak visual density
    fig.update_traces(marker=dict(size=6, opacity=0.7), selector=dict(marker_symbol='circle')) 
    fig.update_traces(marker=dict(size=9, line=dict(width=1.5, color='black')), selector=dict(marker_symbol='square')) 
    
    fig.update_layout(
        width=1100, 
        height=800,
        legend_title_text='Plate & Timepoint',
        xaxis=dict(showticklabels=False, title="UMAP 1"),
        yaxis=dict(showticklabels=False, title="UMAP 2")
    )
    
    # Save the interactive HTML
    save_path = os.path.join(OUTPUT_DIR, f"UMAP5_{config['name'].replace(' ', '_')}.html")
    fig.write_html(save_path)
    print(f"Saved: {save_path}")
    
    # Show the plot
    fig.show()

In [ ]:
import pandas as pd
import numpy as np
import os

# ==========================================
# 0. GLOBAL SETUP
# ==========================================
PROJECT_ROOT = r'E:\groteEdeepprofilerdingen\DataDeepprofiler\finaal'
# Ensure this matches the filename from the previous step
INPUT_CSV = os.path.join(PROJECT_ROOT,"7marchecht", "aggregated_wells_min5.csv") 
OUTPUT_DIR = os.path.join(PROJECT_ROOT, "7marchecht")
CONTROL_LABEL = "no_sgRNA" 

print("Loading raw data...")
df_raw = pd.read_csv(INPUT_CSV)

# Ensure all columns are strings to avoid indexing issues
df_raw.columns = [str(c) for c in df_raw.columns]

# --- CRITICAL CHANGE: Separate Metadata and Features ---
# We keep Cell_Count in metadata so it doesn't get used in math functions
metadata_cols = ['Plate', 'Well_ID', 'Treatment', 'Cell_Count']
feature_cols = [c for c in df_raw.columns if c not in metadata_cols]

# ==========================================
# STEP 1: ACROSS-PLATE STABILITY
# ==========================================
def filter_step1_across_plate_stability(df, features, top_n_to_keep=800):
    ctrls = df[df['Treatment'] == CONTROL_LABEL]
    plate_medians = ctrls.groupby('Plate')[features].median()

    tp_batch_noises = []
    for tp in ['T0', 'T1', 'T2']:
        tp_plates = [p for p in plate_medians.index if p.endswith(tp)]
        if len(tp_plates) > 1:
            noise = plate_medians.loc[tp_plates].std()
            tp_batch_noises.append(noise)
    
    total_batch_noise = pd.concat(tp_batch_noises, axis=1).mean(axis=1)
    
    # Variance check
    variance_filter = df[features].std() > 0.005
    
    stable_features = total_batch_noise[variance_filter].sort_values(ascending=True).head(top_n_to_keep).index.tolist()
    return stable_features

print("\nRunning Step 1: Across-Plate Stability...")
stable_cols = filter_step1_across_plate_stability(df_raw, feature_cols, top_n_to_keep=800)
df_step1 = df_raw[metadata_cols + stable_cols]

# ==========================================
# STEP 2: WITHIN-PLATE CONSISTENCY
# ==========================================
def filter_within_plate_consistency(df, features, top_n_to_keep=500):
    ctrls = df[df['Treatment'] == CONTROL_LABEL]
    within_plate_variation = ctrls.groupby('Plate')[features].std().mean()
    consistent_features = within_plate_variation.sort_values(ascending=True).head(top_n_to_keep).index.tolist()
    return consistent_features

print("Running Step 2: Within-Plate Consistency...")
consistent_features = filter_within_plate_consistency(df_step1, stable_cols, top_n_to_keep=500)
df_step2 = df_step1[metadata_cols + consistent_features]

# ==========================================
# STEP 3: FINAL PLATE CONVERGENCE
# ==========================================
def filter_step3_plate_convergence(df, features, top_n_to_keep=200):
    ctrls = df[df['Treatment'] == CONTROL_LABEL]
    plate_medians = ctrls.groupby('Plate')[features].median()

    tp_noises = []
    for tp in ['T0', 'T1', 'T2']:
        tp_plates = [p for p in plate_medians.index if p.endswith(tp)]
        if len(tp_plates) > 1:
            noise = plate_medians.loc[tp_plates].std()
            tp_noises.append(noise)
    
    avg_plate_noise = pd.concat(tp_noises, axis=1).mean(axis=1)
    plate_blind_features = avg_plate_noise.sort_values(ascending=True).head(top_n_to_keep).index.tolist()
    return plate_blind_features

print("Running Step 3: Final Plate Convergence...")
step3_features = filter_step3_plate_convergence(df_step2, consistent_features, top_n_to_keep=200)
df_step3 = df_step2[metadata_cols + step3_features]

# ==========================================
# STEP 4: REDUNDANCY REMOVAL
# ==========================================
def filter_step4_redundancy(df, features, correlation_threshold=0.9):
    corr_matrix = df[features].corr().abs()
    upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
    to_drop = [column for column in upper.columns if any(upper[column] > correlation_threshold)]
    final_features = [f for f in features if f not in to_drop]
    return final_features

print("Running Step 4: Redundancy Filter...")
non_redundant_features = filter_step4_redundancy(df_step3, step3_features, correlation_threshold=0.9)

# ==========================================
# FINAL SAVE
# ==========================================
df_final_vetted = df_step3[metadata_cols + non_redundant_features]
output_path = os.path.join(OUTPUT_DIR, "vettedcellcounts5_nosd.csv")
df_final_vetted.to_csv(output_path, index=False)

print("\n" + "="*40)
print(f"WORKFLOW COMPLETE")
print(f"Original features: {len(feature_cols)}")
print(f"Final feature count: {len(non_redundant_features)}")
print(f"Master profiles saved to: {output_path}")
print("="*40)

In [ ]:
#thuis:
import pandas as pd
import numpy as np
import umap
import os
from sklearn.preprocessing import StandardScaler
import plotly.express as px

# 1. SETUP
PROJECT_ROOT = r'E:\groteEdeepprofilerdingen\DataDeepprofiler\finaal'
# Using the updated filename from the vetting script
file_path = os.path.join(PROJECT_ROOT,"7marchecht","vettedcellcounts5_nosd.csv")
df = pd.read_csv(file_path)

OUTPUT_DIR = os.path.join(PROJECT_ROOT,"8marchecht", "UMAP_5nosdmarchtest")

if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)

# Ensure column names are strings
df.columns = [str(c) for c in df.columns]

# 2. SELECTION
SELECTED_PLATES = ["PLATE1_T0","PLATE1_T1","PLATE1_T2","PLATE2_T0","PLATE2_T1","PLATE2_T2",
                   "PLATE3_T0","PLATE3_T1","PLATE3_T2","PLATE4_T0","PLATE4_T1","PLATE4_T2",
                   "PLATE5_T0","PLATE5_T1","PLATE5_T2"]

if SELECTED_PLATES:
    df = df[df['Plate'].isin(SELECTED_PLATES)].copy()

# 3. DEFINE CHANNELS
# FIXED: We identify features by checking if the column name is purely numeric
# This excludes 'Plate', 'Well_ID', 'Treatment', 'Cell_Count', and 'Type'
vetted_features = [c for c in df.columns if c.isdigit()]

def get_channel_features(start, end):
    return [f for f in vetted_features if start <= int(f) < end]

plot_configs = [
    {"name": "All Vetted Channels", "indices": vetted_features},
    {"name": "Channel 1", "indices": get_channel_features(0, 1280)},
    {"name": "Channel 2", "indices": get_channel_features(1280, 2560)},
    {"name": "Channel 3", "indices": get_channel_features(2560, 3840)},
    {"name": "Channel 4", "indices": get_channel_features(3840, 5120)},
    {"name": "Channel 5", "indices": get_channel_features(5120, 6400)}
]

# Create a 'Type' column for distinct shapes
df['Type'] = df['Treatment'].apply(lambda x: 'Control' if x.lower() in ["no_sgrna", "nosgrna"] else 'Mutant')

# 4. RUN UMAP FOR EACH CHANNEL
for config in plot_configs:
    if not config['indices']:
        print(f"Skipping {config['name']} (0 features).")
        continue
    
    print(f"Processing interactive UMAP for: {config['name']} ({len(config['indices'])} features)...")
    
    # Scale and Reduce
    X = df[config['indices']].values
    X_scaled = StandardScaler().fit_transform(X)
    
    # UMAP parameters - n_neighbors affects local vs global structure
    reducer = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=42)
    embedding = reducer.fit_transform(X_scaled)
    
    # Temporarily add coordinates to a plotting dataframe
    df_plot = df.copy()
    df_plot['UMAP1'] = embedding[:, 0]
    df_plot['UMAP2'] = embedding[:, 1]
    
    # Create the figure
    fig = px.scatter(
        df_plot, 
        x='UMAP1', 
        y='UMAP2', 
        color='Plate',
        symbol='Type',
        hover_name='Treatment',
        # ADDED: 'Cell_Count' to hover_data so you can check quality during exploration
        hover_data={
            'Plate': True, 
            'Well_ID': True, 
            'Cell_Count': True, 
            'UMAP1': False, 
            'UMAP2': False
        },
        title=f"UMAP: {config['name']} ({len(config['indices'])} features)",
        template='plotly_white',
        symbol_map={'Control': 'square', 'Mutant': 'circle'}
    )
    
    # Tweak visual density
    fig.update_traces(marker=dict(size=6, opacity=0.7), selector=dict(marker_symbol='circle')) 
    fig.update_traces(marker=dict(size=9, line=dict(width=1.5, color='black')), selector=dict(marker_symbol='square')) 
    
    fig.update_layout(
        width=1100, 
        height=800,
        legend_title_text='Plate & Timepoint',
        xaxis=dict(showticklabels=False, title="UMAP 1"),
        yaxis=dict(showticklabels=False, title="UMAP 2")
    )
    
    # Save the interactive HTML
    save_path = os.path.join(OUTPUT_DIR, f"UMAP5_{config['name'].replace(' ', '_')}.html")
    fig.write_html(save_path)
    print(f"Saved: {save_path}")
    
    # Show the plot
    fig.show()